# Inflation model


### Initialization: libraries and initial settings

In [1]:
# ----- Load libraries -----

# Built-in
using Colors, CSV, Dates, DataFrames, Distributions, FileIO, JLD2, XLSX, KernelDensity, PlotlyJS;
pltjs = PlotlyJS

# Custom
include("./code/Metropolis-Within-Gibbs/MetropolisWithinGibbs.jl");
using Main.MetropolisWithinGibbs;

┌ Warning: Kaledio is not available on this system. Julia will be unable to produce any plots.
└ @ PlotlyBase C:\Users\wrc938\.julia\packages\PlotlyBase\NxSlF\src\kaleido.jl:58


### Prepare Results

In [2]:
# ----- Load results from output file -----

res           = load("$(pwd())/res_kuttner_AR2_4_obs_iis.jld2");

nDraws        = res["nDraws"];
burnin        = res["burnin"];
σʸ            = res["σʸ"]';
date          = res["date"];
data          = res["data"] .* σʸ';
distr_α       = res["distr_α"];
chain_θ_bound = res["chain_θ_bound"]; # d, Z, R, c, T, Q, λ, ρ, total
theta_keep = (size(chain_θ_bound, 2)-size(distr_α, 3)+1):size(chain_θ_bound, 2);
chain_θ_bound = chain_θ_bound[:, theta_keep];
MNEMONIC      = res["MNEMONIC"];



data[ismissing.(data)] .= NaN;

par_ind  = res["par_ind"];


par_size = SizeParSsm(sum(par_ind.d),
                                 sum(sum(par_ind.Z)),
                                 sum(sum(par_ind.Z_plus)),
                                 sum(sum(par_ind.Z_minus)),
                                 sum(sum(par_ind.R)),
                                 sum(par_ind.c),
                                 sum(sum(par_ind.T)),
                                 sum(sum(par_ind.Q)),
                                 sum(sum(par_ind.Q_cov)), 
                                 sum(sum(par_ind.λ)),
                                 sum(sum(par_ind.ρ)),
                                 sum(par_ind.d) + sum(sum(par_ind.Z)) + sum(sum(par_ind.Z_plus)) +
                                    sum(sum(par_ind.Z_minus)) + sum(sum(par_ind.R)) +
                                    sum(par_ind.c) + sum(sum(par_ind.T)) + sum(sum(par_ind.Q)) + 
                                    sum(sum(par_ind.Q_cov)) + sum(sum(par_ind.λ)) + sum(sum(par_ind.ρ)));



# ----- Set titles and scales -----

titles = ["Real GDP", "Inflation rate", "SPF Expected inflation", "UoM Inflation Expectations"];
scales = ["log(Y)*100", "Percent", "Percent", "Percent"];

In [3]:
# ----- Get trends and cycles -----

n          = size(data)[2];
println("Number of observables: ", n);
TT         = size(distr_α, 2);
println("Number of time periods: ", TT);
ind_cycles = [4,7,9, 12]
ind_trends = [6,11,14]
println("Cycle indices: ", ind_cycles);

ZΨe = [ones(1, size(distr_α, 3)); 
      chain_θ_bound[1:3, :]]; 





Number of observables: 4
Number of time periods: 174
Cycle indices: [4, 7, 9, 12]


In [4]:
Ψe = zeros(n, TT, size(distr_α, 3));

# Get each observable n's loading on the common states: shape (n, TT, nDraws)
for i=1:size(distr_α, 3)    
    Ψe[:, :, i] = (ZΨe[:, i] .* distr_α[1, :, i]')



end

println("Size of Ψe: ", size(Ψe));


Size of Ψe: (4, 174, 20000)


In [5]:
# idiosyncratic cycle draws
iC = distr_α[ind_cycles, :, :];
iT = distr_α[ind_trends, :, :];


#medians
Ψeᵐ = median(Ψe, dims=3)[:, :, 1]' .* σʸ';
iCᵐ = median(iC, dims=3)[:, :, 1]' .* σʸ';
iTᵐ = median(iT, dims=3)[:, :, 1]' .* [σʸ[1:1]' σʸ[3:4]'];





In [6]:
Ψe95 = zeros(size(Ψeᵐ));
Ψe05 = zeros(size(Ψeᵐ));
Ψe84 = zeros(size(Ψeᵐ));
Ψe16 = zeros(size(Ψeᵐ));

iC95 = zeros(size(iCᵐ));
iC05 = zeros(size(iCᵐ));
iC84 = zeros(size(iCᵐ));
iC16 = zeros(size(iCᵐ));

iT95 = zeros(size(iTᵐ));
iT05 = zeros(size(iTᵐ));
iT84 = zeros(size(iTᵐ));
iT16 = zeros(size(iTᵐ));


for i=1:size(data, 2)
    for j=1:size(data, 1)
        
        Ψe95[j, i] = quantile(Ψe[i, j, :], 0.95) .* σʸ[i];
        Ψe05[j, i] = quantile(Ψe[i, j, :], 0.05) .* σʸ[i];
        Ψe84[j, i] = quantile(Ψe[i, j, :], 0.84) .* σʸ[i];
        Ψe16[j, i] = quantile(Ψe[i, j, :], 0.16) .* σʸ[i];
        

        iC95[j, i] = quantile(iC[i, j, :], 0.95) .* σʸ[i];
        iC05[j, i] = quantile(iC[i, j, :], 0.05) .* σʸ[i];
        iC84[j, i] = quantile(iC[i, j, :], 0.84) .* σʸ[i];
        iC16[j, i] = quantile(iC[i, j, :], 0.16) .* σʸ[i];
               

         if i <2
            iT95[j, i] = quantile(iT[i, j, :], 0.95) .* σʸ[i];
            iT05[j, i] = quantile(iT[i, j, :], 0.05) .* σʸ[i];
            iT84[j, i] = quantile(iT[i, j, :], 0.84) .* σʸ[i];
            iT16[j, i] = quantile(iT[i, j, :], 0.16) .* σʸ[i];
         elseif i>2
            iT95[j, i-1] = quantile(iT[i-1, j, :], 0.95) .* σʸ[i];
            iT05[j, i-1] = quantile(iT[i-1, j, :], 0.05) .* σʸ[i];
            iT84[j, i-1] = quantile(iT[i-1, j, :], 0.84) .* σʸ[i];
            iT16[j, i-1] = quantile(iT[i-1, j, :], 0.16) .* σʸ[i];
         end
      
    end
end

Ψeᵐ_inf = copy(Ψeᵐ)
Ψe95_inf = copy(Ψe95)
Ψe05_inf = copy(Ψe05)
Ψe84_inf = copy(Ψe84)
Ψe16_inf = copy(Ψe16);


In [7]:
# ----- Dates: add h dates to date -----

date=[date[i] for i=1:length(date)];
max_h = size(Ψeᵐ)[1] - size(date)[1];

for hz=1:max_h
    
    last_month = Dates.month(date[end]);
    last_year  = Dates.year(date[end]);
    new_month  = copy(last_month) + 3;
    new_year   = copy(last_year);
    
    if last_month + 1 > 12
        new_year  += 1;
        new_month  = 3;
    end
    
    new_entry = Dates.lastdayofquarter(Date(new_year, new_month, 1));
    date      = vcat(date, DateTime(new_entry));
end

In [8]:
# Colour

c1 = "rgba(0, 48, 158, .75)"; #"rgba(0, 0, 158, .7)";
c2 = "rgba(255, 0, 0, .75)";
c3 = "rgba(255, 190, 0, .75)";

---

### Charts

#### Introduction

#### Historical decomposition

In [9]:
figures = Array{Any}(undef, 8);

for i=1:4
    trace1 = pltjs.bar(;x=date[1:end-max_h], y=Ψeᵐ[1:end-max_h, i], name="Output Gap", marker_color=c1, showlegend=i==i);
    trace3 = pltjs.bar(x=date[1:end-max_h], y=iCᵐ[1:end-max_h, i], name="Idiosyncratic Cycle", marker_color=c3, showlegend=i==i);
    trace4 = pltjs.scatter(x=date[1:end-max_h], y=Ψeᵐ[1:end-max_h, i]+iCᵐ[1:end-max_h, i], name="Total Cycle", mode="lines", line=attr(width=1.4, color="black"), showlegend=i==i)

    databar = [trace1, trace3, trace4];
    layout  = pltjs.Layout(;title=titles[i], titlefont_size=12,
                           xaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), nticks=20, tickangle=-90),
                           yaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), title=scales[i]),
                           barmode="relative", 
                           bargap=0,
                           bargroupgap=0);

    figures[i] = pltjs.plot(databar, layout);
end

fig = [figures[1]]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2,font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")




fig = [figures[2];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end
    
# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")


fig = [figures[3];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end
    
# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")




fig = [figures[4];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end
    
# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

# Inflation+Employment Model

### Prepare Results

In [10]:
# ----- Load results from output file -----

res           = load("$(pwd())/res_okun_kuttner_AR2_6_obs_iis.jld2");

nDraws        = res["nDraws"];
burnin        = res["burnin"];
σʸ            = res["σʸ"]';
date          = res["date"];
data          = res["data"] .* σʸ';
distr_α       = res["distr_α"];
chain_θ_bound = res["chain_θ_bound"]; # d, Z, R, c, T, Q, λ, ρ, total
theta_keep = (size(chain_θ_bound, 2)-size(distr_α, 3)+1):size(chain_θ_bound, 2);
chain_θ_bound = chain_θ_bound[:, theta_keep];
MNEMONIC      = res["MNEMONIC"];



data[ismissing.(data)] .= NaN;

par_ind  = res["par_ind"];


par_size = SizeParSsm(sum(par_ind.d),
                                 sum(sum(par_ind.Z)),
                                 sum(sum(par_ind.Z_plus)),
                                 sum(sum(par_ind.Z_minus)),
                                 sum(sum(par_ind.R)),
                                 sum(par_ind.c),
                                 sum(sum(par_ind.T)),
                                 sum(sum(par_ind.Q)),
                                 sum(sum(par_ind.Q_cov)), 
                                 sum(sum(par_ind.λ)),
                                 sum(sum(par_ind.ρ)),
                                 sum(par_ind.d) + sum(sum(par_ind.Z)) + sum(sum(par_ind.Z_plus)) +
                                    sum(sum(par_ind.Z_minus)) + sum(sum(par_ind.R)) +
                                    sum(par_ind.c) + sum(sum(par_ind.T)) + sum(sum(par_ind.Q)) + 
                                    sum(sum(par_ind.Q_cov)) + sum(sum(par_ind.λ)) + sum(sum(par_ind.ρ)));



# ----- Set titles and scales -----

titles = ["Real GDP", "Employment", "Unemployment rate", "Inflation rate", "UoM Expected Inflation", "SPF Expected inflation"];
scales = ["log(Y)*100", "log(Y)*100", "Percent", "Percent", "Percent", "Percent"];

In [11]:
# ----- Get trends and cycles -----

n          = size(data)[2];
println("Number of observables: ", n);
TT         = size(distr_α, 2);
println("Number of time periods: ", TT);
ind_cycles = [4,7,10,13,15,18]
ind_trends = [6,9,12,17,20]
println("Cycle indices: ", ind_cycles);

ZΨe = [ones(1, size(distr_α, 3)); 
      chain_θ_bound[1:5, :]]; 





Number of observables: 6
Number of time periods: 174
Cycle indices: [4, 7, 10, 13, 15, 18]


In [12]:
Ψe = zeros(n, TT, size(distr_α, 3));

# Get each observable n's loading on the common states: shape (n, TT, nDraws)
for i=1:size(distr_α, 3)    
    Ψe[:, :, i] = (ZΨe[:, i] .* distr_α[1, :, i]')

end

println("Size of Ψe: ", size(Ψe));


Size of Ψe: (6, 174, 20000)


In [13]:
# idiosyncratic cycle draws
iC = distr_α[ind_cycles, :, :];
iT = distr_α[ind_trends, :, :];


#medians
Ψeᵐ = median(Ψe, dims=3)[:, :, 1]' .* σʸ';
iCᵐ = median(iC, dims=3)[:, :, 1]' .* σʸ';
iTᵐ = median(iT, dims=3)[:, :, 1]' .* [σʸ[1:3]' σʸ[5:6]'];





In [14]:
Ψe95 = zeros(size(Ψeᵐ));
Ψe05 = zeros(size(Ψeᵐ));
Ψe84 = zeros(size(Ψeᵐ));
Ψe16 = zeros(size(Ψeᵐ));

iC95 = zeros(size(iCᵐ));
iC05 = zeros(size(iCᵐ));
iC84 = zeros(size(iCᵐ));
iC16 = zeros(size(iCᵐ));

iT95 = zeros(size(iTᵐ));
iT05 = zeros(size(iTᵐ));
iT84 = zeros(size(iTᵐ));
iT16 = zeros(size(iTᵐ));


for i=1:size(data, 2)
    for j=1:size(data, 1)
        
        Ψe95[j, i] = quantile(Ψe[i, j, :], 0.95) .* σʸ[i];
        Ψe05[j, i] = quantile(Ψe[i, j, :], 0.05) .* σʸ[i];
        Ψe84[j, i] = quantile(Ψe[i, j, :], 0.84) .* σʸ[i];
        Ψe16[j, i] = quantile(Ψe[i, j, :], 0.16) .* σʸ[i];
        

        iC95[j, i] = quantile(iC[i, j, :], 0.95) .* σʸ[i];
        iC05[j, i] = quantile(iC[i, j, :], 0.05) .* σʸ[i];
        iC84[j, i] = quantile(iC[i, j, :], 0.84) .* σʸ[i];
        iC16[j, i] = quantile(iC[i, j, :], 0.16) .* σʸ[i];
               

         if i <4
            iT95[j, i] = quantile(iT[i, j, :], 0.95) .* σʸ[i];
            iT05[j, i] = quantile(iT[i, j, :], 0.05) .* σʸ[i];
            iT84[j, i] = quantile(iT[i, j, :], 0.84) .* σʸ[i];
            iT16[j, i] = quantile(iT[i, j, :], 0.16) .* σʸ[i];
         end

           if i > 4
            iT95[j, i-1] = quantile(iT[i-1, j, :], 0.95) .* σʸ[i];
            iT05[j, i-1] = quantile(iT[i-1, j, :], 0.05) .* σʸ[i];
            iT84[j, i-1] = quantile(iT[i-1, j, :], 0.84) .* σʸ[i];
            iT16[j, i-1] = quantile(iT[i-1, j, :], 0.16) .* σʸ[i];
         end

      
    end
end

Ψeᵐ_empinf  = copy(Ψeᵐ)
Ψe95_empinf = copy(Ψe95)
Ψe05_empinf = copy(Ψe05)
Ψe84_empinf = copy(Ψe84)
Ψe16_empinf = copy(Ψe16);


In [15]:
# ----- Dates: add h dates to date -----

date=[date[i] for i=1:length(date)];
max_h = size(Ψeᵐ)[1] - size(date)[1];

for hz=1:max_h
    
    last_month = Dates.month(date[end]);
    last_year  = Dates.year(date[end]);
    new_month  = copy(last_month) + 3;
    new_year   = copy(last_year);
    
    if last_month + 1 > 12
        new_year  += 1;
        new_month  = 3;
    end
    
    new_entry = Dates.lastdayofquarter(Date(new_year, new_month, 1));
    date      = vcat(date, DateTime(new_entry));
end

In [16]:
# Colour

c1 = "rgba(0, 48, 158, .75)"; #"rgba(0, 0, 158, .7)";
c2 = "rgba(255, 0, 0, .75)";
c3 = "rgba(255, 190, 0, .75)";

---

### Charts

#### Introduction

#### Historical decomposition

In [17]:
figures = Array{Any}(undef, 8);

for i=1:6
    trace1 = pltjs.bar(;x=date[1:end-max_h], y=Ψeᵐ[1:end-max_h, i], name="Output Gap", marker_color=c1, showlegend=true);
    trace3 = pltjs.bar(x=date[1:end-max_h], y=iCᵐ[1:end-max_h, i], name="Idiosyncratic Cycle", marker_color=c3, showlegend=true);
    trace4 = pltjs.scatter(x=date[1:end-max_h], y=Ψeᵐ[1:end-max_h, i]+iCᵐ[1:end-max_h, i], name="Total Cycle", mode="lines", line=attr(width=1.4, color="black"), showlegend=true);

    databar = [trace1, trace3, trace4];
    layout  = pltjs.Layout(;title=titles[i], titlefont_size=12,
                           xaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), nticks=20, tickangle=-90),
                           yaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, titlefont=attr(size=10), title=scales[i]),
                           barmode="relative", 
                           bargap=0,
                           bargroupgap=0);

    figures[i] = pltjs.plot(databar, layout);
end

fig = [figures[1]; ]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")




fig = [figures[2];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")


fig = [figures[3];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")

fig = [figures[4];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")



fig = [figures[5];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")



fig = [figures[6];]


# Bars
fig.plot.layout["barmode"] = "relative";
fig.plot.layout["bargap"]  = 0.02;

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 250;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:1
    fig.plot.layout["annotations"][i][:font][:size] = 10;
end

# Legend
fig.plot.layout["legend"] = attr(orientation="h", y=-0.25, x=0.2, font=attr(size=10))
display(fig)
#savefig(fig, "./img/historical_decomposition.pdf", format="pdf")

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

data: [
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "bar with fields marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, bargap, barmode, height, legend, margin, width, xaxis1, and yaxis1"

In [18]:

inf_color = "#B33A3A"                 # median line (red tone)
band68_inf   = "rgba(179, 58, 58, 0.30)" # inner band (red)
band90_inf  = "rgba(179, 58, 58, 0.15)" # outer band (red)

y_lims = (-6, 6);

# --- Reusable layout (set limits if you want) ---


base_layout = Layout(
    template="plotly_white", hovermode="x unified",
    width=900, height=420, margin=attr(l=60, r=20, t=60, b=55),
    xaxis=attr(
        title="Time", showgrid=true, gridcolor="rgba(0,0,0,0.08)",
        zeroline=false, showline=true, linecolor="black", linewidth=1, mirror=true
    ),
    yaxis=attr(
        title="Pct.", showgrid=true, gridcolor="rgba(0,0,0,0.08)",
        zeroline=true, showline=true, linecolor="black", linewidth=1, mirror=true,
        range=[y_lims...]    # comment this line out to auto-scale
    ),
    legend=attr(orientation="h", x=0.0, y=1.02, yanchor="bottom", bgcolor="rgba(0,0,0,0)")
)

# ============= Business Cycle (with 90% & 68% bands, boxed axes) =============
plt_cycle = Plot([
    # 90% band: lower first (invisible), then upper filled to previous
    scatter(x=date, y=Ψe05_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_inf[1:end-max_h, 1], mode="lines", name="90% CI",
            fill="tonexty", fillcolor=band90_inf, line=attr(color="rgba(0,0,0,0)")),

    # 68% band: lower first (invisible), then upper filled to previous
    scatter(x=date, y=Ψe16_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_inf[1:end-max_h, 1], mode="lines", name="68% CI",
            fill="tonexty", fillcolor=band68_inf, line=attr(color="rgba(0,0,0,0)")),

    # Median line
    scatter(x=date, y=Ψeᵐ_inf[1:end-max_h, 1], mode="lines", name="Est. gap Model 1",
            line=attr(color=inf_color, width=2.5))
],base_layout)

display(plt_cycle)




data: [
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, hovermode, legend, margin, template, width, xaxis, and yaxis"



In [19]:
empinf_color = "#2A6EA6"               # median line
band68_emp   = "rgba(42,110,166,0.30)" # inner band
band90_emp   = "rgba(42,110,166,0.15)" # outer band



plt_empinf = Plot([
    # 90% band: lower (invisible) then upper filled to previous
    scatter(x=date, y=Ψe05_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_empinf[1:end-max_h, 1], mode="lines", name="90% CI",
            fill="tonexty", fillcolor=band90_emp, line=attr(color="rgba(0,0,0,0)")),

    # 68% band: lower (invisible) then upper filled to previous
    scatter(x=date, y=Ψe16_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_empinf[1:end-max_h, 1], mode="lines", name="68% CI",
            fill="tonexty", fillcolor=band68_emp, line=attr(color="rgba(0,0,0,0)")),

    # Median
    scatter(x=date, y=Ψeᵐ_empinf[1:end-max_h, 1], mode="lines",
            name="Est. gap Model 2",
            line=attr(color=empinf_color, width=2.5))
],
base_layout)
display(plt_empinf)


data: [
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, hovermode, legend, margin, template, width, xaxis, and yaxis"



In [20]:
# Keep your blue palette for the Inflation-only series


fig_both = Plot([
    # --- Inflation-only ribbons ---
    scatter(x=date, y=Ψe05_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_inf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band90_inf, line=attr(color="rgba(0,0,0,0)")),
    scatter(x=date, y=Ψe16_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_inf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band68_inf, line=attr(color="rgba(0,0,0,0)")),

    # --- Emp+Inf ribbons  ---
    scatter(x=date, y=Ψe05_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_empinf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band90_emp, line=attr(color="rgba(0,0,0,0)")),
    scatter(x=date, y=Ψe16_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_empinf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band68_emp, line=attr(color="rgba(0,0,0,0)")),

    # --- Medians on top ---
    scatter(x=date, y=Ψeᵐ_inf[1:end-max_h, 1], mode="lines",
            name="Est. gap Model 1", line=attr(color=inf_color, width=2.5)),
    scatter(x=date, y=Ψeᵐ_empinf[1:end-max_h, 1], mode="lines",
            name="Est. gap Model 2", line=attr(color=empinf_color, width=2.5))
],base_layout)
display(fig_both)


data: [
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, hovermode, legend, margin, template, width, xaxis, and yaxis"



In [21]:
## add HP filter output gap for comparison
include("code/filters.jl");
y = (data[1:end-max_h, 1]);

hp_gap = HP_filter(y);

# ============= gaps
plt_cycle = Plot([

        # --- Inflation-only ribbons ---
    scatter(x=date, y=Ψe05_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_inf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band90_inf, line=attr(color="rgba(0,0,0,0)")),
    scatter(x=date, y=Ψe16_inf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_inf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band68_inf, line=attr(color="rgba(0,0,0,0)")),

    # --- Emp+Inf ribbons  ---
    scatter(x=date, y=Ψe05_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe95_empinf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band90_emp, line=attr(color="rgba(0,0,0,0)")),
    scatter(x=date, y=Ψe16_empinf[1:end-max_h, 1], mode="lines",
            line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
    scatter(x=date, y=Ψe84_empinf[1:end-max_h, 1], mode="lines", showlegend=false,
            fill="tonexty", fillcolor=band68_emp, line=attr(color="rgba(0,0,0,0)")),

    # --- Medians on top ---
    scatter(x=date, y=Ψeᵐ_inf[1:end-max_h, 1], mode="lines",
            name="Est. gap Model 1", line=attr(color=inf_color, width=2.5)),
    scatter(x=date, y=Ψeᵐ_empinf[1:end-max_h, 1], mode="lines",
            name="Est. gap Model 2", line=attr(color=empinf_color, width=2.5)),
        
    # HP benchmark
    scatter(x=date[1:end-max_h], y=hp_gap, name="HP Gap", line=attr(width=2.5, color="#000000", dash="dash")),

],
base_layout)

data: [
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, and y",
  "scatter with fields fill, fillcolor, line, mode, showlegend, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, name, type, x, and y"
]

layout: "layout with fields height, hovermode, legend, margin, template, width, xaxis, and yaxis"

